In [19]:
import pandas as pd
import numpy as np
import time
import re
df = pd.read_csv('krisha_data_raw.csv')

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5003 entries, 0 to 5002
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   url                   5003 non-null   object
 1   title                 5003 non-null   object
 2   price                 5003 non-null   object
 3   Балкон                2595 non-null   object
 4   Балкон остеклён       941 non-null    object
 5   Безопасность          2340 non-null   object
 6   Бывшее общежитие      4091 non-null   object
 7   Возможен обмен        4449 non-null   object
 8   Высота потолков       4734 non-null   object
 9   Год постройки         5003 non-null   int64 
 10  Город                 5003 non-null   object
 11  Дверь                 2323 non-null   object
 12  Жилой комплекс        4305 non-null   object
 13  Интернет              1578 non-null   object
 14  Квартира меблирована  2673 non-null   object
 15  Парковка              3192 non-null   

In [21]:
df.drop(columns=['Телефон', 'Пол', 'Балкон остеклён', 'Балкон', 'Интернет', 'Дверь', 'Бывшее общежитие'], inplace=True)

In [22]:
df['price'] = df['price'].str.replace(r'\D', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

In [23]:
df['rooms'] = df['title'].str.extract(r'(\d+)-комнатная').astype(float)

floor_data = df['title'].str.extract(r'(\d+)/(\d+)\s+этаж')
df['current_floor'] = floor_data[0].astype(float)
df['total_floors'] = floor_data[1].astype(float)


In [24]:
df['area_m2'] = df['Площадь'].str.extract(r'(\d+\.?\d*)').astype(float)

In [25]:
df.drop(columns='Площадь', inplace=True)

In [26]:
df.drop(columns='Этаж', inplace=True)

In [27]:
df['Тип дома'] = df['Тип дома'].fillna(df['Тип дома'].mode()[0])

In [28]:
df['Санузел'] = df['Санузел'].fillna(df['Санузел'].mode()[0])

In [29]:
df['Высота потолков'] = df['Высота потолков'].str.replace(r'[^\d.]', '', regex=True)
df['Высота потолков'] = df['Высота потолков'].astype(float)
mean_m = df['Высота потолков'].mean()
df['Высота потолков'].fillna(mean_m, inplace = True)

C:\Users\loser\AppData\Local\Temp\ipykernel_17016\1804091221.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Высота потолков'].fillna(mean_m, inplace = True)


In [30]:
df['Квартира меблирована'] = df['Квартира меблирована'].fillna('без мебели')
df['Состояние квартиры'] = df['Состояние квартиры'].fillna('Не указано')
df['Парковка'] = df['Парковка'].fillna('Во дворе/не указано')
df['Безопасность'] = df['Безопасность'].fillna(df['Безопасность'].mode()[0])

In [31]:
df.drop(columns='Возможен обмен', inplace=True)

In [32]:
df['current_floor'] = df['current_floor'].fillna(df['current_floor'].median())
df['total_floors'] = df['total_floors'].fillna(df['total_floors'].median())

In [33]:
df['Жилой комплекс'].fillna('Панельный дом/Не указан', inplace=True)

C:\Users\loser\AppData\Local\Temp\ipykernel_17016\605677421.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Жилой комплекс'].fillna('Панельный дом/Не указан', inplace=True)


In [34]:
def extract_clean_address(title):
    if not isinstance(title, str):
        return "N/A"
    
    if '·' in title:
        parts = title.split('·')
        address_block = parts[-1].strip() 
    else:
        address_block = title

    clean_addr = re.sub(r'^.*?этаж\s*,\s*', '', address_block)
    clean_addr = re.sub(r'^.*?м²\s*,\s*', '', clean_addr)
    
    if ' — ' in clean_addr:
        slogans = ['цена', 'срочно', 'торг', 'акция', 'уступка', 'Горячая цена ! Туран ресторан !', 'САМАЯ НИЗКАЯ ЦЕНА', 'Горячая цеенааа',
        'Горящий вариант ️', 'СРОЧНО!!! УПАКОВАННАЯ', 'СРОООООЧНО', 'СРОЧНО!!! КЛЮЧИ НА РУКАХ!!!', 'ГОРЯЧИЙ ВАРИАНТ', 'суперлокация', 'супер', ] #can be added other trash slogans
        addr_parts = clean_addr.split(' — ')
        if any(s in addr_parts[1].lower() for s in slogans):
            clean_addr = addr_parts[0]
            
    return clean_addr.strip(' ,.!')

df['address'] = df['title'].apply(extract_clean_address)

In [35]:
valid_districts = ['Алматы', 'Есильский', 'Сарыарка', 'Сарайшык', 'Байконур', 'Нура']

df['Город'] = df['Город'].str.replace(r'^Астана,\s*', '', regex=True)
df['Город'] = df['Город'].str.replace(r'\s*р-н\s*', '', regex=True)

df['Город'] = df['Город'].apply(lambda x: x if x in valid_districts else 'Астана')

df = df.rename(columns={'Город': 'Район'})

In [ ]:

# def process_data(df_user):
#     """
#     Optimized geocoding function for Astana housing data.
#     Implements O(u) optimization and robust 429 error handling.
#     """
#     # 1. Configuration & District Fallbacks
#     district_coords = {
#         'Алматы': (51.150, 71.500),
#         'Байконур': (51.145, 71.465),
#         'Есильский': (51.120, 71.430),
#         'Нура': (51.100, 71.380),
#         'Сарыарка': (51.170, 71.410),
#         'Сарайшык': (51.117, 71.565),
#         'Астана': (51.133, 71.433)
#     }

#     df = df_user

#     # Initialize Nominatim with a custom user_agent
#     geolocator = Nominatim(user_agent="astana_housing_mapper_v3", timeout=10)

#     # RateLimiter handles the 429 errors automatically with retries and a 1.5s delay
#     geocode_service = RateLimiter(
#         geolocator.geocode,
#         min_delay_seconds=1.5,
#         max_retries=3,
#         error_wait_seconds=5
#     )

#     def fetch_api(query):
#         """Standardized helper for API calls with strict formatting."""
#         if pd.isna(query) or "не указан" in str(query).lower():
#             return np.nan, np.nan
#         try:
#             location = geocode_service(f"{query}, Astana, Kazakhstan")
#             if location:
#                 # Basic validation: ensure coordinates are in the Astana vicinity
#                 if 50.6 <= location.latitude <= 51.6 and 70.9 <= location.longitude <= 71.9:
#                     return location.latitude, location.longitude
#         except Exception:
#             pass
#         return np.nan, np.nan

#     # --- Step 1: Pre-map District columns for all rows ---
#     # This creates the final safety net for every single entry
#     district_series = df['Район'].map(district_coords)
#     df['district_lat'] = district_series.apply(lambda x: x[0] if isinstance(x, tuple) else district_coords['Астана'][0])
#     df['district_lon'] = district_series.apply(lambda x: x[1] if isinstance(x, tuple) else district_coords['Астана'][1])

#     # Initialize target columns
#     df['apartment_lat'] = np.nan
#     df['apartment_lon'] = np.nan
#     df['coord_level'] = "None"

#     # --- Step 2: Optimization - Geocode Unique Complexes ---
#     valid_complexes = [c for c in df['Жилой комплекс'].unique()
#                        if pd.notna(c) and "не указан" not in str(c).lower()]

#     print(f"Geocoding {len(valid_complexes)} unique complexes...")
#     complex_cache = {}
#     for complex_name in valid_complexes:
#         lat, lon = fetch_api(complex_name)
#         if pd.notna(lat):
#             complex_cache[complex_name] = (lat, lon)

#     # Map results back to the main DataFrame
#     complex_mapped = df['Жилой комплекс'].map(complex_cache)
#     df['apartment_lat'] = complex_mapped.apply(lambda x: x[0] if isinstance(x, tuple) else np.nan)
#     df['apartment_lon'] = complex_mapped.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)
#     df.loc[df['apartment_lat'].notna(), 'coord_level'] = "Complex"

#     # --- Step 3: Optimization - Geocode Unique Addresses for MISSING entries ---
#     missing_mask = df['apartment_lat'].isna()
#     addresses_needed = df.loc[missing_mask, 'address'].dropna().unique()

#     print(f"Geocoding {len(addresses_needed)} unique addresses for remaining gaps...")
#     address_cache = {}
#     for addr in addresses_needed:
#         lat, lon = fetch_api(addr)
#         if pd.notna(lat):
#             address_cache[addr] = (lat, lon)

#     # Update only the rows where complex geocoding failed
#     address_mapped = df.loc[missing_mask, 'address'].map(address_cache)
#     df.loc[missing_mask, 'apartment_lat'] = address_mapped.apply(lambda x: x[0] if isinstance(x, tuple) else np.nan)
#     df.loc[missing_mask, 'apartment_lon'] = address_mapped.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)

#     # Update coord_level for successful address hits
#     df.loc[(missing_mask) & (df['apartment_lat'].notna()) & (df['coord_level'] == "None"), 'coord_level'] = "Address"

#     # --- Step 4: Final Fallback to District Coordinates ---
#     still_missing = df['apartment_lat'].isna()
#     df.loc[still_missing, 'apartment_lat'] = df.loc[still_missing, 'district_lat']
#     df.loc[still_missing, 'apartment_lon'] = df.loc[still_missing, 'district_lon']
#     df.loc[still_missing, 'coord_level'] = "District"

#     # Save and Return
#     # df.to_csv('krisha_final.csv', index=False)
#     print("Success! data was geocoded")
#     return df

# df = process_data(df)

Geocoding 901 unique complexes...
Geocoding 1413 unique addresses for remaining gaps...
Success! data was geocoded


In [ ]:
df['Район'] = df['Район'].replace('Есильский', 'Есильский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Алматы', 'Алматинский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Байконур', 'Байконурский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Сарыарка', 'Сарыаркинский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Сарайшык', 'Сарыаркинский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Нура', 'Нуринский район, Астана, Казахстан')
df['Район'] = df['Район'].replace('Астана', 'Астана, Казахстан')

In [ ]:
df = df.rename(columns={'Безопасность': 'security', 'Высота потолков': 'ceiling_height', 'Район': 'district', 'Жилой комплекс': 'residential_complex',
                                  'Квартира меблирована': 'furnished', 'Парковка': 'parking', 'Санузел': 'bathroom_type',
                                  'Состояние квартиры': 'apartment_condition', 'Тип дома': 'building_type', 'Год постройки':'year_of_construction'})

In [ ]:
df.to_csv('astana_real_estate_dataset.csv', index=False)